# Load Profile Analyser

Choose a site and period (same import mode as `Optimiser.ipynb`), then compute and display load profile metrics via `srce.load_profile_analysis.analyze_load_profile`.

## Data import -- choose site and period

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

sys.path.append(str(Path.cwd().parent))
from srce.data_import import PROFILES
from srce.data_prep import select_first_n_days, select_date_range, select_representative_weeks
from srce.load_profile_analysis import analyze_load_profile

DATA_DIR = Path.cwd().parent / "data"
print("Known sites:", list(PROFILES))

In [ ]:
# ------------------------------ Choose site and period ------------------------------ #
SITE_NAME = "carmeuse"  # one of: "carmeuse", "montea", "lemahieu"
MODE = "year"           # "year", "days", "range", or "weeks"

N_DAYS = 5                     # used when MODE == "days"
START_DATE = "2024-01-01"      # used when MODE == "range" (inclusive)
END_DATE = "2024-01-06"        # used when MODE == "range" (exclusive)
DAYS_PER_MONTH = 7              # used when MODE == "weeks" -- first N days of each calendar month

if SITE_NAME not in PROFILES:
    raise ValueError(f"Unknown SITE_NAME {SITE_NAME!r}. Known sites: {sorted(PROFILES)}")

DATA_PATH = DATA_DIR / "csvs" / f"merged{SITE_NAME}ANDda_belgium.csv"

if MODE == "year":
    df = pd.read_csv(DATA_PATH, parse_dates=["dates"]).sort_values("dates")
elif MODE == "days":
    df = select_first_n_days(input_path=DATA_PATH, days=N_DAYS)
elif MODE == "range":
    df = select_date_range(input_path=DATA_PATH, start=START_DATE, end=END_DATE)
elif MODE == "weeks":
    df = select_representative_weeks(input_path=DATA_PATH, days_per_month=DAYS_PER_MONTH)
else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Choose 'year', 'days', 'range', or 'weeks'.")

df["price"] = df["price [€/MWh]"]
print(f"{SITE_NAME}: {len(df)} rows, {df['dates'].min()} -> {df['dates'].max()}")
df.head()

## Compute metrics

In [ ]:
metrics = analyze_load_profile(df)

## Summary

In [ ]:
# The site profile columns (con/gen/off/inj) are ENERGY per 15-minute
# interval, in kWh -- not instantaneous power. To get an equivalent average
# power in kW, divide by the interval length in hours (DT_HOURS = 0.25).
DT_HOURS = 0.25

start, end = metrics["date_range"]
print(f"Site: {SITE_NAME}")
print(f"Period covered: {start} -> {end}  ({metrics['n_days']} days)")
print()

print(f"Total offtake:   {metrics['total_kwh'].get('off', 0):,.0f} kWh")
print(f"Total injection: {metrics['total_kwh'].get('inj', 0):,.0f} kWh")
print()

off_peak = metrics["peaks"].get("off")
inj_peak = metrics["peaks"].get("inj")
if off_peak:
    print(f"Peak offtake:    {off_peak['value_kwh']:,.1f} kWh/interval  (= {off_peak['value_kwh'] / DT_HOURS:,.1f} kW)  at  {off_peak['timestamp']}")
if inj_peak:
    print(f"Peak injection:  {inj_peak['value_kwh']:,.1f} kWh/interval  (= {inj_peak['value_kwh'] / DT_HOURS:,.1f} kW)  at  {inj_peak['timestamp']}")

avg_off_kwh = df["off"].mean()
print(f"Average offtake: {avg_off_kwh:,.1f} kWh/interval  (= {avg_off_kwh / DT_HOURS:,.1f} kW)")
print()

self_consumption = metrics.get("self_consumption_ratio")
print(f"Self-consumption ratio: {self_consumption:.1%}" if self_consumption is not None else "Self-consumption ratio: n/a (no on-site generation data)")

## Average daily offtake and injection profile

In [ ]:
qh = metrics["quarter_hour_profile"]

fig = go.Figure()
if "off" in qh.columns:
    fig.add_trace(go.Scatter(x=qh.index, y=qh["off"], mode="lines", name="Offtake"))
if "inj" in qh.columns:
    fig.add_trace(go.Scatter(x=qh.index, y=qh["inj"], mode="lines", name="Injection"))

fig.update_layout(
    title=f"{SITE_NAME.title()} -- average daily offtake / injection profile",
    xaxis_title="Time of day",
    yaxis_title="kWh per 15-min interval",
    template="plotly_white",
    height=450,
)
fig.update_xaxes(nticks=24)
fig.show(renderer="browser")